In [44]:
#####################
## IMPORT
#####################
import os 
import shutil
import json

from ultralytics import YOLO

In [45]:
## Récupérer les fichiers hors de val et train : 
def get_files(path):
    files = []
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            if filename.endswith('.jpg'):
                files.append(filename)
    return files

train_val_files = get_files('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/data/split')

In [46]:
non_used = os.listdir('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test YOLO/nonutilisé/images')
len(non_used)

16267

In [47]:
len([i for i in non_used if i not in train_val_files])
## les deux sont égaux, donc pas de doublon

16267

In [48]:
## Tout est ok : on récupère un sample de 100 images
sample_test = [i for i in non_used if i not in train_val_files][:100]

In [49]:
## On crée un dossier pour les images de test
os.makedirs('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images', exist_ok=True)
## On copie les images dans le dossier
for image in sample_test:
    src = os.path.join('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test YOLO/nonutilisé/images', image)
    dst = os.path.join('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images', image)
    shutil.copy(src, dst)


In [50]:
## création du fichier ground_truth_test.json à partir du json de départ
with open('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/ign25synth_train.json', 'r') as f:
    data = json.load(f)


In [51]:

test_gt =[i for i in data if i['image'].split('/')[-1] in sample_test]

In [52]:
# Modifier le nom de l'image dans chaque entrée
for img in test_gt:
    img['image'] = os.path.join('test', img['image'].split('/')[-1])

In [54]:
## export json gt
with open('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/ground_truth_test.json', 'w') as f:
    json.dump(test_gt, f, indent=4)

In [ ]:
## fonction onversion output YOLO dans le même format
## Format attendu :
'''
{'image': 'test/000146.jpg',
  'groups': [[{'vertices': [[268.7747073036216, 1885.3678675115198],
      [299.32158230346687, 1885.3678675115198],
      [299.32158230346687, 1916.3522425127353],
      [268.7747073036216, 1916.3522425127353],
      [268.7747073036216, 1885.3678675115198]],
     'text': '112',
     'illegible': False,
     'truncated': False}],
   [{'vertices': [[136.95474884478415, 1934.4083697779886],
      [167.9391238449445, 1934.4083697779886],
      [167.9391238449445, 1964.955244778889],
      [136.95474884478415, 1964.955244778889],
      [136.95474884478415, 1934.4083697779886]],
     'text': '111',
     'illegible': False,
     'truncated': False}],]}
'''

def conversion_bbox_vertices(bbox, image_size=2000):
    '''
    Convertit une bbox YOLO en liste des sommets du rectangle en coordonnées absolues.

    Args:
        bbox (str): bbox au format YOLO "classe x_center y_center width height"
        image_size (int): taille de l'image (carrée) pour dénormaliser (par défaut : 2000)

    Returns:
        list: liste des 5 sommets (le dernier répété pour fermer le polygone)
    '''
    coords = bbox.strip().split()[1:-1]  # On ignore la classe et la confiance
    x_center, y_center, width, height = [float(coord) * image_size for coord in coords]

    # Calcul des coins
    x_min = x_center - width / 2
    x_max = x_center + width / 2
    y_min = y_center - height / 2
    y_max = y_center + height / 2

    # Sommets dans l'ordre horaire (ou antihoraire) + fermeture du polygone
    vertices = [
        [x_min, y_min],
        [x_max, y_min],
        [x_max, y_max],
        [x_min, y_max],
        [x_min, y_min]
    ]
    return vertices


#Chaque image dans un dico, avec groups qui contient une liste de liste de dictionnaires. Ces dictionaries contiennent les arrêtes, le texte, et les flags illegible et truncated.
def conversion_yolo_output(yolo_output):
    '''
    Convertit l'output de YOLO dans le format attendu pour le fichier ground_truth_test.json.
    yolo_output : str
        Le chemin vers le fichier de sortie de YOLO
    '''
    txt_preds = [i for i in os.listdir(yolo_output) if i.endswith('.txt')]
    preds = {
        f'{txt.replace("txt","jpg")}' : open(os.path.join(yolo_output, txt), 'r').readlines() for txt in txt_preds
    }
    formated_data =[]
    for img,pred in preds.items():
        groups = []
        for bbox in pred:
            vertices = conversion_bbox_vertices(bbox)
            group = [{
                'vertices': vertices,
                'text':'',  
              #  'illegible': False,  # On peut ajuster selon les besoins
               # 'truncated': False  # On peut ajuster selon les besoins
            }]
            groups.append(group)
        formated_data.append({'image': f'test/{img}', 'groups': groups})

    return formated_data

In [62]:
yolo_output = '/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/runs/detect/predict/labels'
converted_output = conversion_yolo_output(yolo_output)

In [63]:
converted_output

[{'image': 'test/002962.jpg',
  'groups': [[{'vertices': [[145.42929999999998, 1800.9827],
      [171.90189999999998, 1800.9827],
      [171.90189999999998, 1831.6933],
      [145.42929999999998, 1831.6933],
      [145.42929999999998, 1800.9827]],
     'text': ''}],
   [{'vertices': [[989.9820000000001, 1618.2301999999997],
      [1017.158, 1618.2301999999997],
      [1017.158, 1651.4217999999998],
      [989.9820000000001, 1651.4217999999998],
      [989.9820000000001, 1618.2301999999997]],
     'text': ''}],
   [{'vertices': [[172.9193, 492.95560000000006],
      [196.9363, 492.95560000000006],
      [196.9363, 518.5444000000001],
      [172.9193, 518.5444000000001],
      [172.9193, 492.95560000000006]],
     'text': ''}],
   [{'vertices': [[1899.5449, 1779.1067],
      [1928.5071, 1779.1067],
      [1928.5071, 1814.7013],
      [1899.5449, 1814.7013],
      [1899.5449, 1779.1067]],
     'text': ''}],
   [{'vertices': [[1089.5542999999998, 749.156],
      [1115.2657, 749.156],
     

In [64]:
## Export du fichier json 
with open('/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/pred_test.json', 'w') as f:
    json.dump(converted_output, f, indent=4)

In [57]:
from ultralytics import YOLO

model = YOLO('/Users/rolly/Downloads/results(1)/runs/detect/yolo_ign25synth_bbox_finetuning2/weights/best.pt')

In [58]:
preds = model.predict(source='/Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/', save=True, save_txt=True, save_conf=True)


image 1/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000146.jpg: 640x640 85 imgs, 93.1ms
image 2/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000152.jpg: 640x640 3 imgs, 73.5ms
image 3/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000608.jpg: 640x640 300 imgs, 101.5ms
image 4/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000620.jpg: 640x640 5 imgs, 77.1ms
image 5/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000634.jpg: 640x640 93 imgs, 79.8ms
image 6/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/000813.jpg: 640x640 87 imgs, 77.4ms
image 7/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_images/001258.jpg: 640x640 267 imgs, 78.7ms
image 8/100 /Users/rolly/Documents/10-19_Université_et_scolarité/python_s2/projet/test_i

In [ ]:
## installation : https://github.com/icdar-maptext/evaluation/tree/main?tab=readme-ov-file#conda-installation
! python3 evaluation/eval.py --gt evaluation/json/ground_truth_test.json --pred evaluation/json/pred_test.json --task det 

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pyeditdistance-1.0.1-py3-none-any.whl.metadata (2.9 kB)
Using cached numpy-1.26.2-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 26.5 MB/s eta 0:00:00a 0:00:01
Using cached pyeditdistance-1.0.1-py3-none-any.whl (4.9 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for shapely (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [140 lines of output]
      <string>:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
      Could not find geos-config executable. Either append the path to geos-config to PATH or manually provide the include_dirs, library_dirs, libraries and